# 3. Late fusion of the text and image models

Late (decision-level) fusion combines the *outputs* of the two independently trained models instead of their raw inputs. A small neural network takes each model's probability and predicted label and makes the final decision.

**Pairing.** The questionnaire and the face datasets come from different people, so there is no real subject with both modalities. The 116 test questionnaires (notebook 1) and the 116 test images (notebook 2) are paired by position to simulate multimodal subjects.

**Ground truth.** Following the thesis, a simulated subject is labelled *anxious* if either modality's **true** label indicates anxiety, and *not anxious* only when both are negative.

In [ ]:
import os
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")  # hide TensorFlow start-up logs
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential, load_model

from preprocessing import load_face, to_model_input

MODELS = Path("../models")
OUTPUTS = Path("../outputs")
CLASSES = ["No anxiety", "Anxiety"]
tf.random.set_seed(11)

## Build the fusion dataset

In [ ]:
text = pd.read_csv(OUTPUTS / "text_predictions.csv")
image = pd.read_csv(OUTPUTS / "image_predictions.csv")
assert len(text) == len(image), "both test sets must have the same size"

fusion = pd.DataFrame(
    {
        "p_text": text["p_anxiety"],
        "text_pred": text["anxiety_pred"],
        "p_image": image["p_anxiety"],
        "image_pred": image["anxiety_pred"],
        "anxiety": text["anxiety_true"] | image["anxiety_true"],
    }
)
FEATURES = ["p_text", "text_pred", "p_image", "image_pred"]

print(fusion.shape)
print(fusion["anxiety"].map(dict(enumerate(CLASSES))).value_counts())
fusion.head()

## Train the fusion network

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    fusion[FEATURES], fusion["anxiety"], test_size=0.2, random_state=11,
    # stratify only when each class has at least two samples
    stratify=fusion["anxiety"] if fusion["anxiety"].value_counts().min() >= 2 else None,
)

fusion_model = Sequential([Input(shape=(len(FEATURES),)), Dense(64, activation="relu"), Dense(1, activation="sigmoid")])
fusion_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
fusion_model.fit(X_train, y_train, epochs=50, batch_size=8, verbose=0)

y_pred = (fusion_model.predict(X_test, verbose=0)[:, 0] > 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=CLASSES, zero_division=0))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", xticklabels=CLASSES, yticklabels=CLASSES, cmap="Blues")
plt.title("Late fusion - test set")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Comparison with single modalities and a simple rule

In [ ]:
test = fusion.loc[X_test.index]
candidates = {
    "Text model only": test["text_pred"],
    "Image model only": test["image_pred"],
    "OR rule (either model says anxiety)": test["text_pred"] | test["image_pred"],
    "Fusion network": pd.Series(y_pred, index=test.index),
}
pd.DataFrame(
    {
        name: {"Accuracy": accuracy_score(y_test, pred), "F1": f1_score(y_test, pred, zero_division=0)}
        for name, pred in candidates.items()
    }
).T.round(3)

In [ ]:
fusion_model.save(MODELS / 'fusion_model.keras')

## End-to-end prediction for one person

Given a face image and the seven anxiety answers of the DASS-21, run both models and fuse their outputs.

In [ ]:
with open(MODELS / "text_model.pkl", "rb") as f:
    text_model = pickle.load(f)
image_model = load_model(MODELS / "image_model.keras")


def predict_person(image_path, answers):
    answers = pd.DataFrame([answers])
    p_text = 1 - text_model.predict_proba(answers)[0, 0]  # 1 - P(normal)
    p_image = image_model.predict(to_model_input([load_face(image_path)]), verbose=0)[0, 1]
    features = pd.DataFrame(
        [[p_text, int(p_text > 0.5), p_image, int(p_image > 0.5)]], columns=FEATURES
    )
    p_fused = fusion_model.predict(features, verbose=0)[0, 0]
    return p_text, p_image, p_fused


answers = {
    "Dryness_of_mouth": 1,
    "Breathing_difficulty": 2,
    "Experienced_trembling": 2,
    "Worried_about_panic_and_make_a_fool_of_myself": 1,
    "Close_to_panic": 2,
    "Aware_of_the_action_of_heart_in_the_absence_of_physical_exertion": 0,
    "Scared_without_any_good_reason": 3,
}
p_text, p_image, p_fused = predict_person(image.loc[0, "path"], answers)

print(f"Text model  P(anxiety) = {p_text:.2f}")
print(f"Image model P(anxiety) = {p_image:.2f}")
print(f"Fused       P(anxiety) = {p_fused:.2f} -> {CLASSES[int(p_fused > 0.5)]}")